Saving all coclusters data in .ods formate

"""
 File Name: Spectral_Coclustering Analysis
 Project: Time Series Forecasting and Clustered Feature Analysis
 Author: Naeem Ullah
 Date: October 2025


───────────────────────────────────────────────────────────────
DESCRIPTION
───────────────────────────────────────────────────────────────
This script implements a **Spectral Co-Clustering–based pipeline** for
unsupervised pattern discovery in time-series air-quality data (e.g., ozone levels).

The code:
1. Loads and cleans raw `.ods` time-series data.
2. Generates lagged and rolling statistical features for temporal context.
3. Determines the optimal number of clusters using the **Elbow Method**.
4. Applies **Spectral Co-Clustering** to jointly group:
      • Rows → time periods with similar dynamics  
      • Columns → features (lags, rolling stats) with correlated behavior
5. Saves each discovered co-cluster to a separate `.ods` file
   for detailed downstream analysis or visualization.

───────────────────────────────────────────────────────────────
PIPELINE OVERVIEW
───────────────────────────────────────────────────────────────
📥 **Step 1 — Data Loading**
   • Reads `.ods` input (e.g., contaminacion_2015_2023.ods)
   • Converts `FECHA_HORA` to datetime and sets it as the index.

⚙️ **Step 2 — Feature Engineering**
   • Rolling statistics: mean, standard deviation, skewness (24-hour window)
   • Lag features: 24 hourly lags of ozone concentration.

📊 **Step 3 — Cluster Optimization**
   • Uses K-Means inertia and the Elbow Method to estimate
     the optimal number of clusters automatically.

🔗 **Step 4 — Spectral Co-Clustering**
   • Applies scikit-learn’s `SpectralCoclustering` to identify biclusters
     representing correlated subsets of features and time segments.

💾 **Step 5 — Export Results**
   • Each co-cluster (row group × feature group) is saved as an `.ods` file
     in the folder `cocluster_outputs/`, preserving the target column.

───────────────────────────────────────────────────────────────
INPUT
───────────────────────────────────────────────────────────────
• `contaminacion_2015_2023.ods`
   Must contain:
   - `FECHA_HORA`: datetime column in `%d/%m/%Y %H:%M` format
   - `ALJARAFE-O3-AT_IN`: target variable (ozone concentration)
   - Other pollutant or meteorological variables (optional)

───────────────────────────────────────────────────────────────
OUTPUT
───────────────────────────────────────────────────────────────
Saved `.ods` files in directory: `cocluster_outputs/`
Example:
   cocluster_0_0.ods
   cocluster_0_1.ods
   cocluster_1_0.ods
   ...

Each file contains:
   • Subset of features in the same co-cluster
   • Corresponding time-indexed ozone values

───────────────────────────────────────────────────────────────
REQUIREMENTS
───────────────────────────────────────────────────────────────
Python >= 3.8  
Libraries: pandas, numpy, matplotlib, scikit-learn, odfpy  

───────────────────────────────────────────────────────────────
HOW TO RUN
───────────────────────────────────────────────────────────────
1. Update `filepath` with the full path to your `.ods` dataset.
2. Run the script:
   ```bash
   python spectral_coclustering_pipeline.py


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import SpectralCoclustering, KMeans
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

#  Load and preprocess the dataset (Exact preprocessing retained)
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")

    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)

    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)

    target_col = 'ALJARAFE-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    #  Add rolling calculations (exactly as requested)
    rolling_window = 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=rolling_window).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=rolling_window).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=rolling_window).skew()

    #  Add lagged features (exactly as requested)
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    df.dropna(inplace=True)  # Ensure no NaN values
    return df, target_col

#  Find optimal number of clusters using the Elbow method
def find_optimal_clusters(data, max_clusters=10):
    ssd = []
    for n_clusters in range(1, max_clusters + 1):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        kmeans.fit(data)
        ssd.append(kmeans.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, max_clusters + 1), ssd, marker='o', linestyle='--')
    plt.xlabel("Number of Clusters")
    plt.ylabel("Sum of Squared Distances (SSD)")
    plt.title("Elbow Method for Optimal Number of Clusters")
    plt.grid()
    plt.show()

    # Automatically find the elbow point
    optimal_clusters = np.diff(ssd, 2).argmin() + 2  # Second derivative method
    print(f"Optimal number of clusters: {optimal_clusters}")
    return optimal_clusters

#  Save each co-cluster to a separate .ods file including the target column
def save_coclusters(df, feature_clusters, instance_clusters, target_col, output_dir="cocluster_outputs"):
    os.makedirs(output_dir, exist_ok=True)  # Ensure directory exists

    for row_cluster_id in range(len(instance_clusters)):
        for col_cluster_id in range(len(feature_clusters)):
            feature_names = feature_clusters[col_cluster_id]

            # Get row indices and feature subset
            row_indices = instance_clusters[row_cluster_id]
            cocluster_df = df[feature_names].iloc[row_indices]

            #  Include the target variable
            cocluster_df[target_col] = df[target_col].iloc[row_indices]

            #  Save to file
            filename = os.path.join(output_dir, f"cocluster_{row_cluster_id}_{col_cluster_id}.ods")
            cocluster_df.to_excel(filename, engine="odf")
            print(f"Saved: {filename}")

#  Main function
def main():
    filepath = r"E:\Abroad period research\Time series forecasting\OneDrive_3_12-18-2024\contaminacion_2015_2023.ods"
    try:
        #  Load and preprocess data (including lagging & rolling statistics)
        df, target_col = load_and_preprocess_data(filepath)
        print("Data loaded and cleaned.")

        #  Select only numeric features (excluding the target column)
        numeric_features = df.select_dtypes(include=[np.number]).drop(columns=[target_col])

        #  Find optimal clusters
        optimal_clusters = find_optimal_clusters(numeric_features)

        print("Applying Spectral Co-Clustering...")
        bicluster = SpectralCoclustering(n_clusters=optimal_clusters, random_state=42)
        bicluster.fit(numeric_features)

        #  Extract feature clusters
        feature_clusters = {i: [] for i in range(optimal_clusters)}
        for i, cluster in enumerate(bicluster.column_labels_):
            feature_clusters[cluster].append(numeric_features.columns[i])

        #  Extract row clusters
        instance_clusters = {i: [] for i in range(optimal_clusters)}
        for i, cluster in enumerate(bicluster.row_labels_):
            instance_clusters[cluster].append(i)

        #  Save co-clustered data
        save_coclusters(df, feature_clusters, instance_clusters, target_col)

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()


Heatmap generation

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# File path
file_path = r"E:\Abroad period research\Time series forecasting\cocluster_outputs\worst performing cluster.ods"

# Load the ODS file (first sheet)
df = pd.read_excel(file_path, engine="odf")

# Convert 'FECHA_HORA' to datetime
df["FECHA_HORA"] = pd.to_datetime(df["FECHA_HORA"], format="%m/%d/%Y")

# Set 'FECHA_HORA' as index
df.set_index("FECHA_HORA", inplace=True)

# Create heatmap 
# The heatmap values come from your actual dataset (they are not random or generated).
# The color scale is based on the min and max values in df.
plt.figure(figsize=(12, 6))
sns.heatmap(df, cmap="coolwarm", annot=False, linewidths=0.5)

# Title and labels
plt.title("Ozone Levels Heatmap Across Lag Features")
plt.xlabel("Lag Features")
plt.ylabel("Date")
plt.xticks(rotation=45)

# Show plot
plt.show()


In [ ]:


# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np

# # File path
# file_path = r"E:\Abroad period research\Time series forecasting\cocluster_outputs\cocluster_4_0.ods"

# # Load the ODS file (first sheet)
# df = pd.read_excel(file_path, engine="odf")

# # Convert 'FECHA_HORA' to datetime
# df["FECHA_HORA"] = pd.to_datetime(df["FECHA_HORA"], format="%m/%d/%Y")

# # Set 'FECHA_HORA' as index
# df.set_index("FECHA_HORA", inplace=True)

# # Convert DataFrame to numpy array for fast processing
# data_array = df.to_numpy()

# # Plot using imshow (fast for large datasets)
# plt.figure(figsize=(14, 8))
# plt.imshow(data_array, aspect='auto', cmap="coolwarm")

# # Add colorbar
# plt.colorbar(label="Ozone Levels")

# # Title and labels
# plt.title("Ozone Levels Heatmap Across Lag Features (Large Dataset)")
# plt.xlabel("Lag Features")
# plt.ylabel("Date")

# # Adjust y-axis labels (use limited labels for readability)
# plt.yticks(np.linspace(0, len(df)-1, 10).astype(int), df.index[::len(df)//10].strftime('%Y-%m-%d'), rotation=45)

# # Show plot
# plt.show()

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# File path
file_path = r"E:\Abroad period research\Time series forecasting\cocluster_outputs\cocluster_2_4.ods"

# Load the ODS file (first sheet)
df = pd.read_excel(file_path, engine="odf")

# Convert 'FECHA_HORA' to datetime
df["FECHA_HORA"] = pd.to_datetime(df["FECHA_HORA"], format="%m/%d/%Y")

# Set 'FECHA_HORA' as index
df.set_index("FECHA_HORA", inplace=True)

# Downsample the data (e.g., every 10th row) for better visualization
df_sampled = df.iloc[::20, :]  # Change 10 to a different number if needed

# Create heatmap
plt.figure(figsize=(14, 8))  # Increase figure size
sns.heatmap(df_sampled, cmap="coolwarm", annot=False, linewidths=0.5)

# Title and labels
plt.title("Ozone Levels Heatmap Across Lag Features (Downsampled)")
plt.xlabel("Lag Features")
plt.ylabel("Date")
plt.xticks(rotation=45)

# Show plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# File path
file_path = r"E:\Abroad period research\Time series forecasting\cocluster_outputs\cocluster_1_1.ods"

# Load the ODS file
df = pd.read_excel(file_path, engine="odf")

# Convert 'FECHA_HORA' to datetime
df["FECHA_HORA"] = pd.to_datetime(df["FECHA_HORA"], format="%m/%d/%Y")

# Set 'FECHA_HORA' as index
df.set_index("FECHA_HORA", inplace=True)

# Resample data weekly for clarity
df_sampled = df.resample('7D').mean().interpolate()

# Alternative 1: Line Plot
plt.figure(figsize=(14, 6))
plt.plot(df_sampled, marker="o", linestyle="-", alpha=0.7)
plt.title("Ozone Levels Trend Over Time", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Ozone Level", fontsize=12)
plt.xticks(rotation=45)
plt.legend(df_sampled.columns, loc="upper right")
plt.grid()
plt.show()

# Alternative 2: Box Plot
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_sampled)
plt.title("Boxplot of Ozone Levels Across Lag Features", fontsize=14)
plt.xlabel("Lag Features", fontsize=12)
plt.ylabel("Ozone Level", fontsize=12)
plt.xticks(rotation=45)
plt.show()

# Alternative 3: Histogram
plt.figure(figsize=(14, 6))
df_sampled.plot(kind='hist', bins=30, alpha=0.7, figsize=(14, 6))
plt.title("Distribution of Ozone Levels", fontsize=14)
plt.xlabel("Ozone Level", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid()
plt.show()

# Alternative 4: Violin Plot
plt.figure(figsize=(14, 6))
sns.violinplot(data=df_sampled, inner="quartile")
plt.title("Violin Plot of Ozone Levels Across Lag Features", fontsize=14)
plt.xlabel("Lag Features", fontsize=12)
plt.ylabel("Ozone Level", fontsize=12)
plt.xticks(rotation=45)
plt.show()

# Alternative 5: Scatter Plot
plt.figure(figsize=(14, 6))
for col in df_sampled.columns:
    plt.scatter(df_sampled.index, df_sampled[col], alpha=0.5, label=col)
plt.title("Scatter Plot of Ozone Levels Over Time", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Ozone Level", fontsize=12)
plt.legend(loc="upper right", fontsize=8)
plt.xticks(rotation=45)
plt.grid()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# File path for co-cluster data
file_path = r"E:\Abroad period research\Time series forecasting\cocluster_outputs\cocluster_2_4.ods"

# Load the ODS file
df_cocluster = pd.read_excel(file_path, engine="odf")

# Display first few rows to understand the structure
print(df_cocluster.head())

# Convert 'FECHA_HORA' to datetime if it exists, otherwise adjust based on your data structure
if 'FECHA_HORA' in df_cocluster.columns:
    df_cocluster["FECHA_HORA"] = pd.to_datetime(df_cocluster["FECHA_HORA"], format="%m/%d/%Y")
    df_cocluster.set_index("FECHA_HORA", inplace=True)

# Resample the data to weekly intervals (or adjust to your preference)
df_sampled = df_cocluster.resample('7D').mean().interpolate()

# ----- Multivariate Line Plot -----
plt.figure(figsize=(14, 6))
plt.plot(df_sampled, marker="o", linestyle="-", alpha=0.7)
plt.title("Ozone Levels and Features Trend Over Time for Co-cluster 2_4", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.xticks(rotation=45)
plt.legend(df_sampled.columns, loc="upper right")
plt.grid()
plt.show()

# # ----- Correlation Heatmap -----
# plt.figure(figsize=(12, 8))
# correlation_matrix = df_sampled.corr()
# sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
# plt.title("Correlation Heatmap of Features in Co-cluster 2_4", fontsize=14)
# plt.xlabel("Features", fontsize=12)
# plt.ylabel("Features", fontsize=12)
# plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# File path for co-cluster data
file_path = r"E:\Abroad period research\Time series forecasting\cocluster_outputs\cocluster_1_1.ods"

# Load the ODS file
df_cocluster = pd.read_excel(file_path, engine="odf")

# Display first few rows to understand the structure
print(df_cocluster.head())

# Convert 'FECHA_HORA' to datetime if it exists, otherwise adjust based on your data structure
if 'FECHA_HORA' in df_cocluster.columns:
    df_cocluster["FECHA_HORA"] = pd.to_datetime(df_cocluster["FECHA_HORA"], format="%m/%d/%Y")
    df_cocluster.set_index("FECHA_HORA", inplace=True)

# Resample the data to weekly intervals (or adjust to your preference)
df_sampled = df_cocluster.resample('7D').mean().interpolate()

# ----- Multivariate Line Plot -----
plt.figure(figsize=(14, 6))

# Use seaborn for smoother lines
sns.set_style("whitegrid")
palette = sns.color_palette("husl", n_colors=len(df_sampled.columns))

for i, column in enumerate(df_sampled.columns):
    plt.plot(df_sampled.index, df_sampled[column], linestyle="-", linewidth=2, label=column, color=palette[i])

plt.title("Ozone Levels and Features Trend Over Time for Co-cluster 2_4", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.xticks(rotation=45)
plt.legend(loc="upper right", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)  # Light dashed grid
plt.show()


BAr chart for Cluster importance 

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Sample data
data = {
    "Cluster ID": range(1, 27),
    "MAE": [11.0751, 5.2368, 13.0474, 10.2336, 14.6219, 11.5259, 12.1129, 4.1357, 13.9638, 8.4784, 18.0044, 12.7485,
            18.3677, 5.3996, 15.6275, 13.5897, 20.9342, 16.4104, 12.2122, 5.3018, 14.1348, 9.6767, 12.7967, 12.1899,
            14.9023, 6.1266],
    "RMSE": [14.1337, 7.1424, 16.5662, 13.0937, 18.3927, 14.5287, 16.2066, 6.0217, 18.0782, 10.917, 22.8244, 16.3705,
             23.1, 7.8766, 20.0944, 17.8941, 26.1885, 20.4976, 15.5236, 7.4157, 17.5851, 12.6532, 16.3145, 15.1887,
             18.2159, 8.3308]
}

df = pd.DataFrame(data)

# Calculate importance as inverse RMSE (lower RMSE means more important cluster)
df["Importance Score"] = 1 / df["RMSE"]

# Normalize importance scores for color mapping
norm = plt.Normalize(df["Importance Score"].min(), df["Importance Score"].max())
colors = plt.cm.viridis(norm(df["Importance Score"]))  # Use viridis colormap

# Create figure and axis
fig, ax = plt.subplots(figsize=(12, 6))

# Plot bars with correct colors
bars = ax.bar(df["Cluster ID"], df["Importance Score"], color=colors)

# Create a colorbar with correct axes handling
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Importance Score")

# Formatting
ax.set_xlabel("Cluster ID")
ax.set_ylabel("Importance Score (1 / RMSE)")
ax.set_title("Cluster Importance for Ozone Forecasting")

# Set x-axis ticks to show cluster IDs correctly
ax.set_xticks(df["Cluster ID"])
ax.set_xticklabels(df["Cluster ID"], rotation=90)

plt.show()
